# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` and plotly are installed
!pip install mlcroissant plotly

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in Croissant (record sets, fields, columns) are referenced via their `@id` fields. This provides an unambiguous reference throughout your workflow.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s)")

for rset in record_sets:
    print(f"\nRecord Set: {rset.name} (id: {rset.id})")
    print(f"  Description: {rset.description}")
    print("  Fields:")
    for fld in rset.fields:
        typ = fld.data_type if hasattr(fld, 'data_type') else fld.type
        print(f"    - {fld.name} (id: {fld.id}, type: {typ})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into a DataFrame
all_dataframes = {}
record_set_ids = [rset.id for rset in record_sets]
for rset in record_sets:
    recs = list(dataset.records(record_set=rset.id))
    df = pd.DataFrame(recs)
    all_dataframes[rset.id] = df

# Print the columns (fields) available in the first record set
main_rs_id = record_sets[0].id
print(f"Columns in record set '{record_sets[0].name}' (id: {main_rs_id}):")
print(all_dataframes[main_rs_id].columns.tolist())
all_dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: e.g. filtering numeric fields, normalizing, grouping, outlier removal. For demonstration, we'll choose a numeric field (e.g. protein molecular weight, or "MW") and a grouping/categorical field.

In [ ]:
# ------- User configuration: Set field @ids here based on overview above ----------
# Example: For demonstration, let's find a typical numeric and a grouping field
# Please change these if different field names/@ids are discovered above!

# Typically in a protein dataset there may be 'MW'/'MolecularWeight' and 'description'/'Gene' etc.
# Let's automatically pick likely candidates from columns for demonstration:
import re
main_df = all_dataframes[main_rs_id]
colnames = {col.lower():col for col in main_df.columns}

numeric_field_candidates = [c for c in main_df.columns if pd.api.types.is_numeric_dtype(main_df[c])]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0] # Just use the first one for demo
else:
    possible = [c for c in main_df.columns if re.search(r"mw|weight|score|count|abundance|number|mass", c, re.I)]
    numeric_field_id = possible[0] if possible else main_df.columns[0]

group_field_candidates = [c for c in main_df.columns if pd.api.types.is_string_dtype(main_df[c]) and c != numeric_field_id]
group_field = group_field_candidates[0] if group_field_candidates else main_df.columns[0]

print(f"Using numeric_field (@id): '{numeric_field_id}'")
print(f"Using group_field (@id): '{group_field}'")

# --- Filter ---
threshold = main_df[numeric_field_id].mean()  # Filter above mean for example
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# --- Normalize ---
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

# --- GroupBy ---
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution for the chosen numeric field and, if applicable, box plots grouped by the categorical/group field.

In [ ]:
import plotly.express as px

# Distribution histogram
fig = px.histogram(main_df, x=numeric_field_id, nbins=40, title=f"Distribution of {numeric_field_id}")
fig.show()

# Box plot grouped by group_field (show up to 20 groups for clarity)
if group_field in main_df.columns and main_df[group_field].nunique() < 30:
    fig2 = px.box(main_df, x=group_field, y=numeric_field_id, title=f"{numeric_field_id} by {group_field}")
    fig2.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We extracted data using Croissant `@id` references, filtered and normalized numeric fields (such as protein abundances or molecular weights), grouped records by categorical fields, and visualized distributions to gain insights into the dataset.

The workflow shown here can be adapted to any Croissant-structured dataset to quickly spin up a reproducible, standards-based EDA environment.

Feel free to further customize analysis steps for your particular biological or computational use case!